# Issue #8 — horizon-specific models and stable ensembles

This notebook reproduces the leakage-safe comparison of standard model families. It uses the frozen rolling-origin contract, tunes 30- and 60-minute models independently, and does not replace the production model.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from gridtoev.horizon_benchmark import run_horizon_benchmark

## Run the frozen-contract benchmark

The final test is read only if a fully frozen candidate first clears every development gate.

In [2]:
result = run_horizon_benchmark()
report = result.report
{
    'report': result.report_path.relative_to(ROOT).as_posix(),
    'development_winner': report['selection']['development_winner'],
    'production_decision': report['selection']['production_decision'],
    'active_model': report['selection']['active_model'],
    'final_test_accessed': report['final_test']['accessed'],
}

{'report': 'benchmarks/horizon_models/benchmark_report.json',
 'development_winner': 'ensemble__v1.1.0__extra_trees_residual',
 'production_decision': 'retain_v1.1.0',
 'active_model': 'v1.1.0',
 'final_test_accessed': False}

## Candidate comparison

Lower MAE is better. The gate column also checks horizon and fold stability, not merely the aggregate score.

In [3]:
registry = pd.read_csv(result.registry_path)
registry.sort_values('rolling_mae_mwh')[
    ['candidate', 'kind', 'rolling_mae_mwh', 'rolling_30m_mae_mwh',
     'rolling_60m_mae_mwh', 'fold_mae_std_mwh', 'improvement_vs_v1_1',
     'development_gate_passed']
].head(12)

,candidate,kind,rolling_mae_mwh,rolling_30m_mae_mwh,rolling_60m_mae_mwh,fold_mae_std_mwh,improvement_vs_v1_1,development_gate_passed
12,ensemble__v1.1.0__extra_trees_residual,oof_horizon_weighted_ensemble,9.751480,7.220199,12.282761,5.181926,0.012176,False
13,ensemble__v1.1.0__extra_trees_residual_conserv...,oof_horizon_weighted_ensemble,9.791661,7.222707,12.360616,5.218253,0.008106,False
10,ensemble__v1.1.0__hist_gradient_boosting_residual,oof_horizon_weighted_ensemble,9.826253,7.233817,12.418690,5.354108,0.004602,False
11,ensemble__v1.1.0__hist_gradient_boosting_conse...,oof_horizon_weighted_ensemble,9.843913,7.235665,12.452162,5.339615,0.002813,False
0,v1.1.0,baseline,9.871679,7.235665,12.507692,5.367626,NaN,NaN
17,ensemble__hist_gradient_boosting_conservative_...,oof_horizon_weighted_ensemble,10.202149,7.559586,12.844712,5.347235,-0.033477,False
15,ensemble__hist_gradient_boosting_residual__ext...,oof_horizon_weighted_ensemble,10.213097,7.539527,12.886667,5.404069,-0.034586,False
16,ensemble__hist_gradient_boosting_residual__ext...,oof_horizon_weighted_ensemble,10.237937,7.558517,12.917357,5.467487,-0.037102,False
18,ensemble__hist_gradient_boosting_conservative_...,oof_horizon_weighted_ensemble,10.238055,7.587101,12.889009,5.414905,-0.037114,False
14,ensemble__hist_gradient_boosting_residual__his...,oof_horizon_weighted_ensemble,10.273414,7.583002,12.963826,5.563098,-0.040696,False


## Why the best candidate was rejected

A small average gain is insufficient when it comes from only a subset of time folds.

In [4]:
winner = report['selection']['development_winner']
baseline_mae = report['baselines']['v1.1.0']['overall']['mae']
winner_mae = report['candidates'][winner]['overall']['mae']
gate = report['selection']['development_gate']
{
    'baseline_mae_mwh': round(baseline_mae, 4),
    'winner_mae_mwh': round(winner_mae, 4),
    'improvement_percent': round(100 * gate['aggregate_mae_improvement_fraction'], 2),
    'horizon_improvements': gate['horizon_mae_improvement_fraction'],
    'fold_improvements': gate['fold_mae_improvement_fraction'],
    'failed_checks': gate['failed_checks'],
}

{'baseline_mae_mwh': 9.8717,
 'winner_mae_mwh': 9.7515,
 'improvement_percent': 1.22,
 'horizon_improvements': {'30': 0.002137436217000252,
  '60': 0.017983412111617955},
 'fold_improvements': {'train_fold_1': 0.003909939023075391,
  'train_fold_2': -0.04491263910783327,
  'train_fold_3': -0.02647404650904701,
  'validation_fold': 0.04949971224197697},
 'failed_checks': ['aggregate_improvement', 'fold_stability']}

## Feature-family ablations

These runs show whether the strongest individual learner benefits from a narrower, more interpretable feature family.

In [5]:
pd.DataFrame([
    {
        'feature_family': name,
        'feature_count': values['feature_count'],
        'rolling_mae_mwh': values['overall']['mae'],
        'rolling_30m_mae_mwh': values['by_horizon']['30']['mae'],
        'rolling_60m_mae_mwh': values['by_horizon']['60']['mae'],
    }
    for name, values in report['feature_family_ablations']['results'].items()
]).sort_values('rolling_mae_mwh')

,feature_family,feature_count,rolling_mae_mwh,rolling_30m_mae_mwh,rolling_60m_mae_mwh
0,all_features,119,10.311389,7.596996,13.025782
2,renewable_grid_state,53,10.520569,7.764205,13.276933
4,temporal_dynamics,66,10.543950,7.708956,13.378944
5,all_without_dispatch_history,109,10.594639,7.846171,13.343107
3,demand_market_interconnector,37,10.905213,7.899209,13.911217
1,persistence_history,11,11.283535,8.124058,14.443011
